<a href="https://colab.research.google.com/github/eeeewyz/agent/blob/main/7_evals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# M4 Agentic AI - Adding a component-level eval to the research workflow

## 1. Introduction

In the previous graded lab (M3), you built a tool-using research agent that carried out a workflow of three steps:

1. Search the web for information.  
2. Reflect on its output.  
3. Publish a clear HTML report.  

Now, in this ungraded lab, you are going to focus on evaluating **one component of that workflow**: the *research step*.  

Instead of generating essays and refining them, here you will design a **component-level evaluation** to check the quality of sources returned by the research step.  

The evaluation will compare the URLs retrieved by the agent against a **predefined list of preferred domains** (e.g., `arxiv.org`, `nature.com`, `nasa.gov`).  

This allows you to quantify whether the system is pulling information from trustworthy sources, using an **objective, per-example ground truth evaluation**.


### 1.1. Lab overview

In the video, Andrew showed a case where web search results were of **poor quality**, making it difficult to trust the information retrieved.  
Building on that example, in this lab you will evaluate the reliability of sources by comparing them against a **predefined list of preferred domains**.

For this evaluation, we’ll focus on the topic *“recent developments in black hole science”*, one of the examples highlighted in the course.  
The idea is to verify whether the web search tool is returning sources from preferred domains, and to quantify the ratio of preferred vs. total results.

This evaluation will be implemented as a single function that performs an **objective, per-example check**. It will:

* Parse the Tavily output (our web search tool).  
* Identify which URLs belong to the list of **preferred domains**.  
* Compute the ratio of preferred vs. total retrieved sources.  
* Return both a boolean flag (**PASS/FAIL**) and a Markdown-formatted summary that can be embedded directly into reports.  

<img src="M4-UGL-1.png" width="70%">  


### 1.2. 🎯 Learning outcomes

You will learn how to:

* Write a function that can check the search results of a web search API for **preferred sources**.  
* Create an evaluation to verify if your sources come from your **preferred domains**.  
* Add a **component-level evaluation** to the web search function.  


核心思路是：

先定义一组你信任的 preferred domains（优选域名），比如权威科研机构、期刊、大学等。
让 Tavily 搜索“recent developments in black hole science”。
然后检查搜索结果中的 URL：
有多少来自优选域名
总共有多少来源
计算 优选来源数 / 总来源数
最后输出：
一个 PASS / FAIL
一个 Markdown 格式的评估报告

## 2. Setup: Import libraries and load environment

As in previous labs, you start by importing the required libraries and initializing your environment.

In [ ]:
# =========================
# Imports
# =========================

# --- Standard library
from datetime import datetime
import json
import re

# --- Third-party ---
from aisuite import Client

# --- Local / project ---
import research_tools
import utils

client = Client()

## 3. Research Step – `find_references`

In the graded lab, the function you implemented both **searched the web and wrote a draft report** in one step.

Here, we split the web search functionality into a separate function called `find_references`. This allows you to evaluate the search results independently of the writing and reflection steps, which we will leave out from this lab since we are only focusing on the output of the web search step.

Notice two key differences from the graded lab implementation:

* This new function uses **AISuite**, which automatically manages the tool calls for you (instead of writing manual tool-calling code with the OpenAI SDK).  
* The function also informs the LLM of the **current date**, which helps improve relevance for time-sensitive queries.  

The role of `find_references` is to **gather external information** from tools such as **Arxiv**, **Tavily**, and **Wikipedia**.  
Because the quality of these results directly shapes the outputs of the graded lab, this is the stage where you can apply **evaluation methods** — for example, checking whether the returned URLs come from your list of **preferred domains**.  

原来的 graded lab 里是：

搜索网页 → 直接写 draft report

现在改成单独一个函数：

find_references()

它只负责：

调 Arxiv、Tavily、Wikipedia 等工具找资料
返回搜索结果 / URL
不负责后面的写作和 reflection

这样做的目的就是：

可以单独评估 web search 这个 component，而不受后面写作质量影响。

另外这里还强调两个变化：

用 AISuite 自动管理 tool calling，不再手写 OpenAI SDK 的工具调用流程。
把当前日期告诉 LLM，让它处理“最近、最新”这类查询时更准确。

最后，find_references 的输出可以拿来做 component-level evaluation，比如检查：

返回的 URL
vs
preferred / gold domains

看它搜到的来源是不是你希望的高质量网站。

#  根据 return_messages 决定返回内容
        # False：只返回最终搜索/研究结果
        # True：同时返回 content 和 messages
        return (content, messages) if return_messages else content

    except Exception as e:
        # 7. 如果模型调用、tool calling 等过程发生异常
        # 不让程序直接崩溃，而是返回错误信息
        return f"[Model Error: {e}]"



解释：





如果用户要求 return_messages=True：
        返回 content + messages

    否则：
        只返回 content


如果上面任何地方发生错误：

    把错误保存成 e

    ↓

    返回 "[Model Error: 错误具体内容]"

In [ ]:
def find_references(
    task: str,
    model: str = "openai:gpt-4o",
    return_messages: bool = False
):
#     return_messages: bool = False

#     参数叫 return_messages
#     希望它是 bool
#     默认值是 False
#     所以普通调用：
#     result = find_references("Find recent papers about RAG")
#     因为你没有传 return_messages，默认：
#     return_messages = False



    """
    使用外部搜索工具完成 research task。
    可调用：
    - arxiv：搜索论文
    - tavily：搜索网页
    - wikipedia：百科信息
    """


    # 1. 构造给 LLM 的 prompt
    # 告诉模型：
    # - 自己有哪些工具可以使用
    # - 当前要完成什么 task
    # - 当前日期是什么，方便处理“最新/最近”这类问题
    prompt = f"""
    You are a research function with access to:
    - arxiv_tool: academic papers
    - tavily_tool: general web search (return JSON when asked)
    - wikipedia_tool: encyclopedic summaries

    Task:
    {task}

    Today is {datetime.now().strftime('%Y-%m-%d')}.
    """.strip()

    # 2. 把 prompt 放入 messages
    # 当前这里只有一条 user message
    messages = [
        {"role": "user", "content": prompt}
    ]

    # 3. 定义允许 LLM 调用的工具
    tools = [
        research_tools.arxiv_search_tool,
        research_tools.tavily_search_tool,
        research_tools.wikipedia_search_tool,
    ]

    try:
        # 4. 调用 LLM
        # LLM 会根据 task 自动决定：
        # - 是否调用工具
        # - 调哪个工具
        # - 调几次工具
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=tools,

            # auto = 让模型自己判断是否需要调用工具
            tool_choice="auto",

            # 最多允许进行 5 轮
            # LLM → tool → LLM → tool ...
            max_turns=5,
        )

        # 5. 取得最终 LLM 输出
        content = response.choices[0].message.content

        # 6. 根据 return_messages 决定返回内容
#         return (content, messages) if return_messages else content
#         这是 Python 的三元表达式 / 条件表达式。
#         完整写法其实就是：
#         if return_messages:
#             return (content, messages)
#         else:
#             return content
#         上面两段代码完全等价。

        return (content, messages) if return_messages else content

    except Exception as e:
        # 7. 如果模型调用、tool calling 等过程发生异常
        # 不让程序直接崩溃，而是返回错误信息
        return f"[Model Error: {e}]"

Run the following cell to test the research function.  
This task will retrieve two recent papers on developments in black hole science and display the results.

In [ ]:
research_task = "Find 2 recent papers about recent developments in black hole science"
research_result = find_references(research_task)

utils.print_html(
    research_result,
    title="Research Function Output"
)

## 4. Evaluation Step – Preferred Domains

Not all sources retrieved by web search are equally reliable.  
In this lab, we focus on **just one step from the previous graded lab** — the `find_references` research step — and show how to design a **component-level evaluation** that checks whether the returned domains belong to a predefined list of **preferred domains**.  

This is an example of an **objective evaluation with a clear per-example ground truth**.  
As a reminder from the lecture, recall the two axes of evaluation: along these axes we are working in the **upper-left quadrant** — objective evaluations with explicitly defined ground truth applied at the level of each example.

<img src='M4-UGL-1-Evaluations.png' width='80%'>


### Why component-level evaluations?

As Andrew mentioned in the lecture:  

- If the problem lies in web search (usually the **first step** in a graded lab workflow), rerunning the *entire* pipeline (search → draft → reflect) every time can be **expensive** and noisy.  
- Small improvements in web search quality may be hidden by randomness introduced by later components.  
- By evaluating the web search *alone*, you get a **clearer signal** of whether that component is improving.  

Component-level evals are also efficient when multiple teams are working on different pieces of a system: each team can optimize its own component using a clear metric, without needing to run or wait for full end-to-end tests.  

### How do we evaluate?

Our evaluation here is **objective**, and so can be evaluated using code. It has an example-specific ground truth - the list of preferred sources for this black hole query. To build the eval, you will:

1. Extract the URLs returned by Tavily.  
2. Compare them against a predefined list of **preferred domains** (e.g., `arxiv.org`, `nature.com`, `nasa.gov`).  
3. Compute the **ratio of preferred vs. total results**.  
4. Return a **PASS/FAIL flag** along with a Markdown-formatted summary.  

This provides a reproducible, low-cost metric that tells us whether the research component — and only this step from the graded lab — is pulling from trusted sources.



这页是在讲：怎么单独评估 research / web search 这个 component。

步骤很简单：

提取 Tavily 返回的所有 URL
和预先定义的优质网站名单比较
比如 arxiv.org
nature.com
nasa.gov
计算优质来源占比
preferred sources / total results
根据阈值给出 PASS / FAIL
同时输出一个 Markdown 总结

例如：

搜索返回 10 个结果
其中 7 个来自 preferred domains

ratio = 7 / 10 = 0.7

raw 搜索结果
   ↓
正则提取 URL
   ↓
逐个拿出 domain
   ↓
是否属于 TOP_DOMAINS？
   ↓
统计 preferred_count
   ↓
ratio = preferred_count / total
   ↓
ratio >= 0.4 ?
   ↓
PASS / FAIL
   ↓
返回 flag + report

.gov、.edu、.org 这些。

它们叫 顶级域名（Top-Level Domain, TLD），可以粗略理解成网站“属于哪一类”。

最常见的：

.gov：government，政府机构
例如 nasa.gov、noaa.gov
.edu：education，教育机构
美国很多大学用 .edu，例如 mit.edu、stanford.edu
.org：organization，组织机构
常见于非营利组织、学术组织，但现在并不是只有非营利机构才能注册
例如 wikipedia.org、arxiv.org
.com：commercial，商业机构
例如 nature.com、springer.com
.net：network，最早主要用于网络服务机构，现在用途很广

输入举例：

raw 在这里就是一个普通的 字符串 str，里面包含研究函数最终输出的文本，而这段文本里可能夹着多个 URL。

比如最简单的样子：

raw = """
Here are some useful sources:

1. NASA black hole research
https://www.nasa.gov/black-holes/

2. Recent paper from arXiv
https://arxiv.org/abs/2401.12345

3. Wikipedia overview
https://en.wikipedia.org/wiki/Black_hole
"""

输出举例：
return flag, report

这两个分别是：

flag：最终是否通过
report：为什么通过/失败的详细报告

比如前面算出来：

ratio = 0.6
min_ratio = 0.4
flag = ratio >= min_ratio

那么：

flag

就是：

True

表示这次评估 PASS。

In [ ]:
# 1. 定义“优质 / 推荐域名”集合
# 后面会判断搜索结果里的 URL 是否来自这些网站
TOP_DOMAINS = {
    # 通用参考网站 / 学术机构 / 出版商
    "wikipedia.org", "nature.com", "science.org", "sciencemag.org", "cell.com",
    "mit.edu", "stanford.edu", "harvard.edu", "nasa.gov", "noaa.gov", "europa.eu",

    # CS / AI 常见论文平台与会议
    "arxiv.org", "acm.org", "ieee.org", "neurips.cc", "icml.cc", "openreview.net",

    # 其他可靠学术来源
    "elifesciences.org", "pnas.org", "jmlr.org", "springer.com", "sciencedirect.com",

    # 针对当前任务额外加入的网站
    "pbs.org", "nova.edu", "nvcc.edu", "cccco.edu",

    # 知名编程学习网站
    "codecademy.com", "datacamp.com"
}


def evaluate_tavily_results(TOP_DOMAINS, raw: str, min_ratio=0.4):
    """
    评估搜索结果中的 URL，有多少来自 preferred domains。

    参数：
        TOP_DOMAINS:
            推荐域名集合

        raw:
            Tavily / research function 返回的原始文本
            文本中可能包含多个 URL

        min_ratio:
            通过评估所需的最低优质来源比例
            默认 0.4，也就是 40%

    返回：
        (flag, markdown_report)

        flag:
            True  -> PASS
            False -> FAIL

        markdown_report:
            Markdown 格式的详细评估报告
    """


    # --------------------------------------------------
    # 2. 从 raw 文本中提取所有 URL
    # --------------------------------------------------

    # 创建正则表达式：
    # 匹配以 http:// 或 https:// 开头的网址
    url_pattern = re.compile(
        r'https?://[^\s\]\)>\}]+',
        flags=re.IGNORECASE
    )

    # 在 raw 中找出所有匹配的 URL
    # 返回一个 list
    urls = url_pattern.findall(raw)


    # --------------------------------------------------
    # 3. 如果一个 URL 都没找到，直接判 FAIL
    # --------------------------------------------------

    if not urls:
        return False, """### Evaluation — Tavily Preferred Domains
No URLs detected in the provided text.
Please include links in your research results.
"""


    # --------------------------------------------------
    # 4. 初始化统计变量
    # --------------------------------------------------

    # 总共有多少个 URL
    total = len(urls)

    # 有多少个 URL 来自 preferred domains
    preferred_count = 0

    # 保存每一个 URL 的具体判断结果
    details = []


    # --------------------------------------------------
    # 5. 逐个检查每个 URL
    # --------------------------------------------------

    for url in urls:

        # 例如：
        # https://www.nasa.gov/black-hole
        #
        # split("/") 后大概是：
        # ["https:", "", "www.nasa.gov", "black-hole"]
        #
        # 所以 [2] 就是 domain
        domain = url.split("/")[2]


        # 判断 domain 是否包含 TOP_DOMAINS 中的任意一个域名
        #
        # 例如：
        # domain = "www.nasa.gov"
        #
        # "nasa.gov" in "www.nasa.gov"
        # -> True
        preferred = any(
            td in domain
            for td in TOP_DOMAINS
        )


        # 如果是推荐域名，计数 +1
        if preferred:
            preferred_count += 1


        # 把每个 URL 的判断结果记录下来
        #
        # preferred == True
        # -> ✅ PREFERRED
        #
        # preferred == False
        # -> ❌ NOT PREFERRED
        details.append(
            f"- {url} → "
            f"{'✅ PREFERRED' if preferred else '❌ NOT PREFERRED'}"
        )


    # --------------------------------------------------
    # 6. 计算 preferred ratio
    # --------------------------------------------------

    # 例如：
    # 一共 10 个 URL
    # 其中 6 个来自优质网站
    #
    # ratio = 6 / 10 = 0.6
    ratio = preferred_count / total if total > 0 else 0.0


    # --------------------------------------------------
    # 7. 和阈值比较
    # --------------------------------------------------

    # 默认：
    # min_ratio = 0.4
    #
    # ratio >= 0.4
    # -> True -> PASS
    #
    # ratio < 0.4
    # -> False -> FAIL
    flag = ratio >= min_ratio


    # --------------------------------------------------
    # 8. 生成 Markdown 格式的报告
    # --------------------------------------------------

    report = f"""
### Evaluation — Tavily Preferred Domains

- Total results: {total}
- Preferred results: {preferred_count}

# :.2% 表示以百分比显示，并保留两位小数
- Ratio: {ratio:.2%}

# :.0% 表示百分比，不保留小数
- Threshold: {min_ratio:.0%}

- Status: {"✅ PASS" if flag else "❌ FAIL"}

**Details:**

# chr(10) 就是换行符 \n
# 把 details 里面的每一行用换行拼起来
{chr(10).join(details)}
"""


    # --------------------------------------------------
    # 9. 返回两个结果
    # --------------------------------------------------

    # flag：
    # True / False
    #
    # report：
    # Markdown 格式的详细报告
    return flag, report

<div style="border:1px solid #93c5fd; border-left:6px solid #3b82f6; background:#dbeafe; border-radius:6px; padding:12px 14px; color:#1e3a8a; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif;">  
<strong>🔎 Why this is an objective evaluation:</strong><br><br>  
Each URL retrieved from Tavily is compared against a predefined list of <em>preferred domains</em> (<code>TOP_DOMAINS</code>):<br>  
• If the domain matches → ✅ PREFERRED<br>  
• Otherwise → ❌ NOT PREFERRED<br><br>  
This yields a clear PASS/FAIL signal depending on whether the ratio of preferred sources exceeds a given threshold.  
Because the ground truth (preferred vs. not preferred) is explicitly defined for each example, the evaluation is both <strong>objective</strong> and <strong>reproducible</strong>.  
</div>


所以它叫 objective evaluation，核心原因是：

判断标准是提前写死的，程序直接按规则计算，不需要人临时主观判断，也不需要 LLM Judge。

而且同样的输入重复跑评估，规则相同，结果也应该相同，所以它还是：

reproducible（可复现的）

Run the cell to display sample preferred domains, the research results, and the evaluation summary (PASS/FAIL with details).

In [ ]:
utils.print_html(json.dumps(list(TOP_DOMAINS)[:4], indent=2), title="Sample Trusted Domains")

utils.print_html("<h3>Research Results</h3>" + research_result, title="Research Results")

flag, report = evaluate_tavily_results(TOP_DOMAINS, research_result)
utils.print_html("<pre>" + report + "</pre>", title="<h3>Evaluation Summary</h3>")

## Try yourself!

Now it’s your turn.  
In this section, you can experiment directly with the **research step** and its **evaluation**:  

* **Topic**: choose a different topic to research.  
* **Preferred domains**: edit or expand the `TOP_DOMAINS` list.  
* **Evaluation ratio**: adjust the `min_ratio` (e.g., 0.4 = at least 40% preferred sources).  

Re-run the cells below after making your edits to see how the evaluation changes.


这里最核心的是这一行：

flag, eval_md = evaluate_tavily_results(
    TOP_DOMAINS,
    research_output,
    min_ratio=0.4
)

它内部大概是在算：

Preferred Ratio=
所有来源数量
preferred domain来源数量
不是让另一个 LLM 主观判断“这些来源好不好”。
	​


In [ ]:
# === 5.1. Try it yourself: topic, ratio & preferred domains ===

# --------------------------------------------------
# ① 设置实验参数
# --------------------------------------------------

# 要研究的主题
topic = "recent developments in black hole science"

# 优选域名占比的最低阈值
# 0.4 = 至少 40% 的来源要来自 preferred domains
min_ratio = 0.4

# 是否运行 reflection
# 这段代码里暂时还没有真正使用它，
# 一般会在后续步骤决定是否对结果进行反思/重写
run_reflection = True


# --------------------------------------------------
# ② 定义 Preferred Domains
# --------------------------------------------------

# 用 set 保存可信/优选域名
# set 的特点：元素不会重复，适合做 membership check
TOP_DOMAINS = {
    "wikipedia.org",
    "nature.com",
    "science.org",
    "arxiv.org",
    "nasa.gov",
    "mit.edu",
    "stanford.edu",
    "harvard.edu"
}


# --------------------------------------------------
# ③ 显示 Preferred Domains
# --------------------------------------------------

import json

utils.print_html(
    # set → list → 排序 → JSON格式字符串
    json.dumps(sorted(list(TOP_DOMAINS)), indent=2),
    title="Sample Preferred Domains"
)


# --------------------------------------------------
# ④ Research：调用搜索工具获取资料
# --------------------------------------------------

# 根据 topic 自动构造搜索任务
research_task = (
    f"Find 2–3 key papers and reliable overviews about {topic}."
)

# 调用 find_references() 搜索相关论文和资料
research_output = find_references(research_task)

# 显示搜索结果
utils.print_html(
    research_output,
    title=f"Research Results on {topic}"
)


# --------------------------------------------------
# ⑤ Component-level Evaluation
# 检查 Web Search 返回来源的质量
# --------------------------------------------------

flag, eval_md = evaluate_tavily_results(
    TOP_DOMAINS,       # 可信域名列表
    research_output,   # Tavily 搜索结果
    min_ratio=min_ratio
)

# flag：
# True  → PASS
# False → FAIL
#
# eval_md：
# Markdown 格式的详细评估结果，
# 例如总 URL 数、preferred URL 数、占比等

utils.print_html(
    eval_md,
    title="Evaluation Summary"
)

## 5. Takeaways

* You just saw how to evaluate the performance of **one component**: the `find_references` research step.  
* Your component-level evaluation checked whether the retrieved URLs were in a predefined list of **preferred domains**.  
* This is an example of an **objective evaluation** with a clear **per-example ground truth**.  
* To build an evaluation set, you could design ~10 prompts covering different topics (astronomy, robotics, finance, etc.) and define preferred domains for each.  
* The percentage of retrieved sources that matched the list of preferred domains provides a useful **metric** to guide improvements, such as adjusting the prompt or tool parameters.  
* This approach is **simpler and cheaper** than evaluating full essays with reflection and rewrites, since you only focus on the web search component.  

<div style="border:1px solid #22c55e; border-left:6px solid #16a34a; background:#dcfce7; border-radius:6px; padding:14px 16px; color:#064e3b; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif;">

🎉 **Congratulations!**  

You designed a **component-level evaluation** that makes your research agent more reliable.  
By directly checking the quality of sources, you introduced a safeguard that is **objective, reproducible, and cost-effective**.  

This aligns with the idea highlighted in Andrew’s lecture: *component-level evaluations* let you test individual pieces of an AI system without the overhead of evaluating the entire pipeline.  

</div>


